# NurseGemma - AI Companion for Nurses

## MedGemma Impact Challenge Submission

**Built by a nurse, for nurses.** As an ICU nurse, I spend 40% of my shift charting instead of caring for patients. NurseGemma is an AI companion that helps explain things to worried families, double-checks medication interactions, and generates handoff reports - so nurses can focus on what matters: **our patients**.

### Modules
1. **Quick Explain** - Medical jargon → plain English for patients & families
2. **Med Helper** - Conversational medication lookup with nursing focus
3. **Shift Sidekick** - SBAR handoff report generation
4. **Clinical Quick Ref** - Lab values, procedures, assessments

### Why This Matters
- **92%** of nurses say EHR negatively impacts job satisfaction
- **40%** of every shift spent on documentation, not patients
- **65%** of hospital patients don't understand their treatment
- **100,000** RNs left the workforce in the past 2 years

---

*Author: AIHeartICU | Powered by MedGemma 1.5*

## Setup & Installation

In [ ]:
# Install required packages
!pip install -q transformers>=4.50.0 accelerate gradio torch huggingface_hub

# Authenticate with HuggingFace (required for MedGemma gated model)
# ============================================================
# SETUP INSTRUCTIONS:
# 1. Go to huggingface.co/settings/tokens - create a READ token
# 2. Go to huggingface.co/google/medgemma-1.5-4b-it - click "Agree and access"
# 3. In this Kaggle notebook: Add-ons > Secrets > Add new secret
#    - Label: HUGGINGFACE_TOKEN
#    - Value: your hf_... token
# 4. Enable GPU: Settings (right panel) > Accelerator > GPU T4 x2
# ============================================================

import os

# Try multiple authentication methods
authenticated = False

# Method 1: Kaggle Secrets (preferred)
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HUGGINGFACE_TOKEN")
    login(token=hf_token)
    print("✓ Successfully authenticated with HuggingFace via Kaggle Secrets!")
    authenticated = True
except Exception as e:
    print(f"Kaggle Secrets method: {e}")

# Method 2: Environment variable (fallback)
if not authenticated:
    try:
        from huggingface_hub import login
        hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
        if hf_token:
            login(token=hf_token)
            print("✓ Successfully authenticated with HuggingFace via environment variable!")
            authenticated = True
    except Exception as e:
        print(f"Environment variable method: {e}")

# Method 3: Already logged in
if not authenticated:
    try:
        from huggingface_hub import whoami
        user = whoami()
        print(f"✓ Already logged in as: {user['name']}")
        authenticated = True
    except:
        pass

if not authenticated:
    print("\n" + "="*60)
    print("⚠ AUTHENTICATION REQUIRED")
    print("="*60)
    print("Please add your HuggingFace token as a Kaggle Secret:")
    print("1. Click 'Add-ons' in the top menu")
    print("2. Click 'Secrets'")
    print("3. Add new secret with:")
    print("   Label: HUGGINGFACE_TOKEN")
    print("   Value: your token from huggingface.co/settings/tokens")
    print("4. Make sure you accepted the license at:")
    print("   huggingface.co/google/medgemma-1.5-4b-it")
    print("="*60)

In [ ]:
import torch
import gradio as gr
from transformers import AutoProcessor, AutoModelForImageTextToText, pipeline
from typing import Optional, List
from dataclasses import dataclass
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Load MedGemma Model

In [ ]:
# Load MedGemma 1.5 4B (latest instruction-tuned version)
# MedGemma 1.5 has improved medical reasoning (+5% MedQA),
# EHR understanding (+22% EHRQA), and lab report extraction (+18% F1)
MODEL_ID = "google/medgemma-1.5-4b-it"

print(f"Loading {MODEL_ID}...")
print("This may take a few minutes on first run.")
print("Note: You must accept the license at huggingface.co/google/medgemma-1.5-4b-it")

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

print(f"\n✓ MedGemma 1.5 loaded successfully!")
print(f"Device: {next(model.parameters()).device}")

## Core Helper Function

In [ ]:
def generate_response(
    system_prompt: str,
    user_message: str,
    max_new_tokens: int = 500,
    temperature: float = 0.7,
) -> str:
    """Generate response from MedGemma"""
    
    messages = [
        {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
        {"role": "user", "content": [{"type": "text", "text": user_message}]},
    ]
    
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)
    
    input_len = inputs["input_ids"].shape[-1]
    
    with torch.inference_mode():
        generation = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=temperature if temperature > 0 else None,
        )
        generation = generation[0][input_len:]
    
    return processor.decode(generation, skip_special_tokens=True)

## Module 1: Quick Explain

**Purpose:** Translate medical jargon into plain language for patients and families.

**Use Case:** A doctor just told a family their loved one has "atrial fibrillation with rapid ventricular response" and they look confused. The nurse needs to explain it simply.

In [ ]:
QUICK_EXPLAIN_SYSTEM = """You are a helpful nursing assistant specialized in patient and family communication.
Your role is to explain medical terms, diagnoses, and procedures in simple, easy-to-understand language.

Guidelines:
- Use plain language appropriate for the specified reading level
- Avoid medical jargon unless explaining it
- Use analogies and comparisons to everyday things
- Be compassionate and reassuring while remaining accurate
- Keep explanations concise but complete
- If something is serious, be honest but gentle
- Suggest follow-up questions the patient might want to ask their doctor"""

def quick_explain(
    term: str,
    reading_level: str = "8th grade",
    audience: str = "family member",
    context: str = ""
) -> str:
    """Explain medical terms in plain language"""
    
    user_message = f"""Please explain the following medical term/condition to a {audience}.
Use a {reading_level} reading level.

Term/Condition: {term}

{"Additional context: " + context if context else ""}

Provide:
1. A simple explanation of what this means
2. Why this might be happening or why it's being done
3. What they can expect
4. Any reassuring points (if appropriate)
5. Questions they might want to ask the doctor"""
    
    return generate_response(QUICK_EXPLAIN_SYSTEM, user_message, max_new_tokens=600)

In [ ]:
# Demo: Quick Explain
print("=" * 60)
print("QUICK EXPLAIN DEMO")
print("=" * 60)

response = quick_explain(
    term="atrial fibrillation",
    reading_level="8th grade",
    audience="worried family member",
    context="The patient was just diagnosed after their heart monitor showed irregular rhythm"
)
print(response)

## Module 2: Med Helper

**Purpose:** Provide nursing-focused medication information including side effects, administration tips, and patient teaching.

**Use Case:** A patient asks "What's this new pill for?" or a nurse needs to quickly review monitoring parameters.

In [ ]:
MED_HELPER_SYSTEM = """You are a medication information assistant for nurses.
Your role is to provide nursing-focused medication information including:
- What the medication is used for (in plain terms)
- Common side effects to monitor
- Key nursing considerations for administration
- Patient teaching points
- Drug interactions when asked

Guidelines:
- Focus on practical nursing information, not pharmacology textbook details
- Highlight what nurses need to watch for
- Provide patient teaching points in simple language
- Always recommend checking with pharmacy for specific patient questions
- Note any high-alert medication considerations"""

def med_info(medication: str, include_interactions: bool = True) -> str:
    """Get nursing-focused medication information"""
    
    user_message = f"""Provide nursing-focused information about: {medication}

Please include:
1. What is this medication used for? (plain language)
2. Common side effects to monitor
3. Key nursing considerations for administration
4. Patient teaching points (in simple language for patient education)
{"5. Common drug interactions to watch for" if include_interactions else ""}

Format clearly with sections."""
    
    return generate_response(MED_HELPER_SYSTEM, user_message, max_new_tokens=700)


def check_interactions(medications: List[str]) -> str:
    """Check for drug-drug interactions"""
    
    user_message = f"""Please check for potential drug interactions between these medications:
{', '.join(medications)}

For each significant interaction found:
1. Which drugs interact
2. What is the interaction (what happens)
3. Clinical significance (mild/moderate/severe)
4. What to monitor
5. Nursing actions to take

If no significant interactions, state that clearly but remind to verify with pharmacy."""
    
    return generate_response(MED_HELPER_SYSTEM, user_message, max_new_tokens=800)

In [ ]:
# Demo: Med Helper - Single Medication
print("=" * 60)
print("MED HELPER DEMO: Metoprolol")
print("=" * 60)

response = med_info("metoprolol")
print(response)

In [ ]:
# Demo: Med Helper - Interaction Check
print("=" * 60)
print("MED HELPER DEMO: Drug Interaction Check")
print("=" * 60)

response = check_interactions(["warfarin", "aspirin", "ibuprofen"])
print(response)

## Module 3: Shift Sidekick

**Purpose:** Generate SBAR handoff reports from clinical notes to ensure safe, complete transitions of care.

**Use Case:** End of shift - the nurse needs to give a clear, organized handoff to the oncoming nurse.

In [ ]:
SHIFT_SIDEKICK_SYSTEM = """You are a nursing handoff assistant that helps generate SBAR reports.

SBAR Format:
- Situation: What is happening with the patient right now?
- Background: What is the clinical context/history?
- Assessment: What do you think the problem is?
- Recommendation: What needs to be done?

Guidelines:
- Be concise but thorough
- Highlight critical information prominently
- Include pending tasks and follow-ups
- Note any changes from previous shift
- Flag any concerning trends or findings"""

def generate_sbar(patient_info: str, style: str = "standard") -> str:
    """Generate SBAR handoff report"""
    
    style_guide = {
        "standard": "Use standard SBAR format with complete sections",
        "detailed": "Use detailed SBAR with subsections and comprehensive information",
        "brief": "Use brief SBAR, highlight only critical points for quick handoff"
    }
    
    user_message = f"""Generate a nursing shift handoff report in SBAR format from this information:

{patient_info}

Instructions: {style_guide.get(style, style_guide['standard'])}

Format as:
**SITUATION**
[Current state]

**BACKGROUND**
[Relevant history and context]

**ASSESSMENT**
[Nursing assessment and concerns]

**RECOMMENDATION**
[What needs to happen next shift]

Also include:
- Critical tasks pending
- Changes from last shift
- Concerning trends to watch"""
    
    return generate_response(SHIFT_SIDEKICK_SYSTEM, user_message, max_new_tokens=1000)

In [ ]:
# Demo: Shift Sidekick - SBAR Generation
print("=" * 60)
print("SHIFT SIDEKICK DEMO: SBAR Report")
print("=" * 60)

sample_patient_info = """
72 y/o male, admitted 3 days ago for community-acquired pneumonia
PMH: HTN, DM2, COPD
Currently on day 3 of Ceftriaxone 1g IV q24h and Azithromycin 500mg IV daily
Vitals trending stable: T 99.2, HR 88, BP 138/82, RR 20, O2 sat 94% on 2L NC
Morning labs: WBC improved to 11.2 from 14.6 yesterday, BMP normal
Patient reports feeling better, eating 50% of meals
Still requiring O2 when ambulating - sats drop to 89% without supplemental O2
IV access good, PIV 20g in right forearm, flushes well, no signs of infiltration
PT/OT consult ordered yesterday, pending evaluation
Wife visited this morning, asking about discharge timeline
Blood cultures from admission still pending final read (preliminary negative)
"""

response = generate_sbar(sample_patient_info, style="standard")
print(response)

## Module 4: Clinical Quick Ref

**Purpose:** Provide instant clinical reference for lab interpretation, procedures, and condition monitoring.

**Use Case:** A lab result comes back and the nurse needs quick interpretation and action guidance.

In [ ]:
CLINICAL_REF_SYSTEM = """You are a clinical reference assistant for nurses.
Your role is to provide quick, accurate clinical information including:
- Normal lab value ranges and interpretation
- Procedure steps and reminders
- Assessment findings and what they indicate
- Clinical pathways and protocols
- Emergency reference information

Guidelines:
- Be accurate and evidence-based
- Provide context for when values are concerning
- Give practical bedside tips
- Always recommend verifying with unit protocols
- Flag emergency situations clearly"""

def interpret_lab(lab_name: str, value: float, unit: str, context: str = "") -> str:
    """Interpret lab value with clinical context"""
    
    user_message = f"""Interpret this lab result:

Lab: {lab_name}
Value: {value} {unit}
{f"Patient context: {context}" if context else ""}

Please provide:
1. Normal range for this lab
2. Is this value normal, high, or low?
3. What this value might indicate clinically
4. What to assess/monitor related to this value
5. When to notify the provider
6. Any nursing interventions to consider"""
    
    return generate_response(CLINICAL_REF_SYSTEM, user_message, max_new_tokens=500)


def what_to_watch(condition: str) -> str:
    """Get monitoring priorities for a condition"""
    
    user_message = f"""For a patient with {condition}, what should I watch for?

Provide:
1. Key assessment findings to monitor
2. Vital sign considerations
3. Signs of improvement
4. Warning signs/red flags
5. When to escalate care
6. Common complications to watch for
7. Patient education points"""
    
    return generate_response(CLINICAL_REF_SYSTEM, user_message, max_new_tokens=600)

In [ ]:
# Demo: Clinical Quick Ref - Lab Interpretation
print("=" * 60)
print("CLINICAL QUICK REF DEMO: Lab Interpretation")
print("=" * 60)

response = interpret_lab(
    lab_name="Potassium",
    value=6.2,
    unit="mEq/L",
    context="Patient has chronic kidney disease, on lisinopril and spironolactone"
)
print(response)

In [ ]:
# Demo: Clinical Quick Ref - What to Watch
print("=" * 60)
print("CLINICAL QUICK REF DEMO: What to Watch")
print("=" * 60)

response = what_to_watch("new onset atrial fibrillation")
print(response)

## Interactive Demo with Gradio

A user-friendly interface for all Nurse Companion modules.

In [ ]:
# Create Gradio Interface

def quick_explain_ui(term, reading_level, audience, context):
    return quick_explain(term, reading_level, audience, context)

def med_helper_ui(medication, check_interactions_opt):
    return med_info(medication, check_interactions_opt)

def interaction_check_ui(meds):
    med_list = [m.strip() for m in meds.split(",")]
    return check_interactions(med_list)

def sbar_ui(patient_info, style):
    return generate_sbar(patient_info, style)

def lab_ui(lab_name, value, unit, context):
    return interpret_lab(lab_name, float(value), unit, context)

def watch_ui(condition):
    return what_to_watch(condition)

# Build the interface
with gr.Blocks(title="Nurse Companion", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""# 🏥 Nurse Companion
    ### Your AI Shift Buddy - Built by a nurse, for nurses
    *Powered by MedGemma*
    """)
    
    with gr.Tabs():
        # Tab 1: Quick Explain
        with gr.Tab("📖 Quick Explain"):
            gr.Markdown("### Explain medical terms in plain language")
            with gr.Row():
                with gr.Column():
                    qe_term = gr.Textbox(label="Medical Term/Condition", placeholder="e.g., atrial fibrillation")
                    qe_level = gr.Dropdown(
                        choices=["5th grade", "8th grade", "high school", "college"],
                        value="8th grade",
                        label="Reading Level"
                    )
                    qe_audience = gr.Dropdown(
                        choices=["worried family member", "the patient directly", "child's parents"],
                        value="worried family member",
                        label="Audience"
                    )
                    qe_context = gr.Textbox(label="Additional Context (optional)", placeholder="Any relevant situation details...")
                    qe_btn = gr.Button("Explain", variant="primary")
                with gr.Column():
                    qe_output = gr.Textbox(label="Explanation", lines=15)
            qe_btn.click(quick_explain_ui, [qe_term, qe_level, qe_audience, qe_context], qe_output)
        
        # Tab 2: Med Helper
        with gr.Tab("💊 Med Helper"):
            gr.Markdown("### Medication information for nurses")
            with gr.Row():
                with gr.Column():
                    med_name = gr.Textbox(label="Medication Name", placeholder="e.g., metoprolol")
                    med_interactions = gr.Checkbox(label="Include interaction information", value=True)
                    med_btn = gr.Button("Get Info", variant="primary")
                with gr.Column():
                    med_output = gr.Textbox(label="Medication Information", lines=15)
            med_btn.click(med_helper_ui, [med_name, med_interactions], med_output)
            
            gr.Markdown("---")
            gr.Markdown("### Check Drug Interactions")
            with gr.Row():
                with gr.Column():
                    int_meds = gr.Textbox(label="Medications (comma-separated)", placeholder="warfarin, aspirin, ibuprofen")
                    int_btn = gr.Button("Check Interactions", variant="secondary")
                with gr.Column():
                    int_output = gr.Textbox(label="Interaction Report", lines=10)
            int_btn.click(interaction_check_ui, int_meds, int_output)
        
        # Tab 3: Shift Sidekick
        with gr.Tab("📋 Shift Sidekick"):
            gr.Markdown("### Generate SBAR handoff reports")
            with gr.Row():
                with gr.Column():
                    sbar_info = gr.Textbox(
                        label="Patient Information",
                        lines=10,
                        placeholder="Enter patient details, vitals, events, labs, etc..."
                    )
                    sbar_style = gr.Dropdown(
                        choices=["standard", "detailed", "brief"],
                        value="standard",
                        label="Report Style"
                    )
                    sbar_btn = gr.Button("Generate SBAR", variant="primary")
                with gr.Column():
                    sbar_output = gr.Textbox(label="SBAR Report", lines=20)
            sbar_btn.click(sbar_ui, [sbar_info, sbar_style], sbar_output)
        
        # Tab 4: Clinical Quick Ref
        with gr.Tab("🔬 Clinical Quick Ref"):
            gr.Markdown("### Lab interpretation & monitoring guides")
            
            gr.Markdown("#### Interpret Lab Values")
            with gr.Row():
                with gr.Column():
                    lab_name = gr.Textbox(label="Lab Name", placeholder="e.g., Potassium")
                    lab_value = gr.Number(label="Value")
                    lab_unit = gr.Textbox(label="Unit", placeholder="e.g., mEq/L")
                    lab_context = gr.Textbox(label="Patient Context (optional)", placeholder="e.g., patient on dialysis")
                    lab_btn = gr.Button("Interpret", variant="primary")
                with gr.Column():
                    lab_output = gr.Textbox(label="Interpretation", lines=12)
            lab_btn.click(lab_ui, [lab_name, lab_value, lab_unit, lab_context], lab_output)
            
            gr.Markdown("---")
            gr.Markdown("#### What to Watch For")
            with gr.Row():
                with gr.Column():
                    watch_condition = gr.Textbox(label="Condition/Diagnosis", placeholder="e.g., new onset heart failure")
                    watch_btn = gr.Button("Get Monitoring Guide", variant="secondary")
                with gr.Column():
                    watch_output = gr.Textbox(label="Monitoring Guide", lines=12)
            watch_btn.click(watch_ui, watch_condition, watch_output)
    
    gr.Markdown("""---
    *Disclaimer: This tool is for educational and reference purposes. Always verify with facility protocols, pharmacy, and clinical judgment. Not a substitute for professional medical advice.*
    
    **Created for the MedGemma Impact Challenge by AIHeartICU**
    """)

# Launch the demo
demo.launch(share=True)

## Impact & Future Directions

### Measured Impact
If deployed, Nurse Companion could help address:
- **40% of shift time** currently spent on documentation
- **65% of patients** who don't understand their treatment
- **40% of nurses** considering leaving the profession

### Future Enhancements
1. **Voice integration** - Hands-free queries at bedside
2. **EHR integration** - Auto-pull patient context
3. **Multi-language support** - 35% of US speaks non-English at home
4. **Specialty modules** - ICU, ER, L&D, Peds-specific content
5. **Fine-tuning** - Train on real nursing scenarios for better accuracy

### Safety Considerations
- All outputs should be verified by the nurse
- Not a replacement for clinical judgment
- Medication info should be confirmed with pharmacy
- Follows facility protocols

---

**Thank you for reviewing Nurse Companion!**

*Built with MedGemma for the MedGemma Impact Challenge*
*By AIHeartICU - An ICU nurse who believes AI should help us care for patients, not create more paperwork.*